Load libraries/packages



In [1]:
import pandas as pd
import numpy as np
import sqlite3


In [2]:
df = pd.read_csv("../DataCo/DataCoSupplyChainDataset.csv", encoding='latin1')
print(f"Data loaded successfull! Shape: {df.shape}")

Data loaded successfull! Shape: (180519, 53)


This data has 53 columns with string, integer and float types.The columns are information of shipping dates, customer information, delivery status, risk, ...

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

There is no null value but there are NaN values

In [4]:
print(f"First few lines of the data:\n {pd.DataFrame(df.head())}")

First few lines of the data:
        Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0                   0           73  Sporting Goods        Caguas  ...   
1                 

Clean column names so they are easy to use in SQL

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")

In [6]:
df.order_date_dateorders = pd.to_datetime(df.order_date_dateorders, format ="%m/%d/%Y %H:%M")
df.shipping_date_dateorders = pd.to_datetime(df.shipping_date_dateorders, format ="%m/%d/%Y %H:%M")

In [7]:
#dropping useless comlumns (customer's information and product_description-null values and product_image)
df = df.drop(columns = ["customer_email", "customer_password", "customer_fname", "customer_lname", "customer_street", "product_description", "product_image"])

Open a connection to local database (if the supply_chain.db is not there, running this line will create a new one)

In [8]:
conn = sqlite3.connect("../data/supply_chain.db")

Write the data from DataCoSupplyChainDataset.csv to the database supply_chain.db

In [9]:
df.to_sql("supply_chain", conn, if_exists='replace', index = False)

180519

In [10]:
pd.read_sql("PRAGMA table_info(supply_chain);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,type,TEXT,0,None,0
1,1,days_for_shipping_real,INTEGER,0,None,0
2,2,days_for_shipment_scheduled,INTEGER,0,None,0
3,3,benefit_per_order,REAL,0,None,0
4,4,sales_per_customer,REAL,0,None,0
5,5,delivery_status,TEXT,0,None,0
6,6,late_delivery_risk,INTEGER,0,None,0
7,7,category_id,INTEGER,0,None,0
8,8,category_name,TEXT,0,None,0
9,9,customer_city,TEXT,0,None,0


In [11]:
# check if datetime format is correct
pd.read_sql("SELECT order_date_dateorders, shipping_date_dateorders FROM supply_chain LIMIT 3;", conn)

,order_date_dateorders,shipping_date_dateorders
0,2018-01-31 22:56:00,2018-02-03 22:56:00
1,2018-01-13 12:27:00,2018-01-18 12:27:00
2,2018-01-13 12:06:00,2018-01-17 12:06:00


In [12]:
conn.close()